## IEC Voter Registration Extraction Approach

Before extracting the data, the IEC Voter Registration Statistics page was inspected to understand how the website delivers its data.

The page does not expose the required municipality and ward registration data directly in the initial HTML. The province, municipality, and ward selections are handled through the IEC's ASP.NET Web Forms interface.

The browser's network activity was therefore inspected to identify what happens when selections are made. This showed that selecting a province triggers an asynchronous POST request to the same IEC page, carrying ASP.NET form-state fields such as `__VIEWSTATE`, `__EVENTVALIDATION`, the selected province, and the event target.

Based on this observation, we will reproduce the website's normal selection process programmatically rather than trying to invent or rely on an undocumented API.

### Extraction flow

`IEC page → KwaZulu-Natal → Municipality → Ward → Registration statistics`

The extraction will:

1. Start an HTTP session with the IEC website.
2. Load the initial page and obtain the required ASP.NET state fields.
3. Select **KwaZulu-Natal** programmatically.
4. Retrieve the available KwaZulu-Natal municipalities.
5. Select each municipality and retrieve its wards.
6. Retrieve the registration statistics available for each ward.
7. Repeat this across the required **2011–2026** period where the IEC source provides the corresponding data.
8. Verify the extracted structure and records.
9. Save the original extracted results to `data/raw/`.

This notebook is limited to **data extraction and verification**. Cleaning, transformation, feature engineering, analysis, and modelling will be handled later in the project.

In [1]:
#1.
# We are extracting IEC voter registration statistics
# for KwaZulu-Natal only, covering 2011 to 2026.
#
# The 2011–2026 period provides a historical time series
# for examining changes in voter registration over time.
#
# The IEC website uses an ASP.NET form, so we will
# reproduce the same province → municipality → ward
# selection process programmatically.
#
# Raw extracted data will be saved in:
# data/raw/


from pathlib import Path
import pandas as pd
import requests
from bs4 import BeautifulSoup

# Project root
PROJECT_ROOT = Path.cwd().parent

# Raw data directory
RAW_DIR = PROJECT_ROOT / "data" / "raw"
RAW_DIR.mkdir(parents=True, exist_ok=True)

# IEC source
IEC_URL = (
    "https://www.elections.org.za/pw/StatsData/"
    "Voter-Registration-Statistics"
)

# Extraction scope
TARGET_PROVINCE = "KwaZulu-Natal"
KZN_PROVINCE_ID = "4"

# Historical period requested
START_YEAR = 2011
CURRENT_YEAR = 2026

print("IEC extraction setup complete.")
print("Source:", IEC_URL)
print("Province:", TARGET_PROVINCE)
print("Province ID:", KZN_PROVINCE_ID)
print("Period:", START_YEAR, "to", CURRENT_YEAR)
print("Raw directory:", RAW_DIR)

IEC extraction setup complete.
Source: https://www.elections.org.za/pw/StatsData/Voter-Registration-Statistics
Province: KwaZulu-Natal
Province ID: 4
Period: 2011 to 2026
Raw directory: C:\Users\Student\Downloads\Big Data\SPU-TEAM-DIRISA\data\raw


In [2]:
#2.
# We first open the IEC page and keep a session active.
# The session is needed because the IEC uses ASP.NET
# state and cookies when moving between selections.

iec_session = requests.Session()

initial_response = iec_session.get(
    IEC_URL,
    timeout=30
)

print("HTTP status:", initial_response.status_code)
print("Content type:", initial_response.headers.get("Content-Type"))
print("Response size:", len(initial_response.content), "bytes")

assert initial_response.status_code == 200, (
    "IEC voter registration page could not be loaded."
)

initial_soup = BeautifulSoup(
    initial_response.text,
    "html.parser"
)

print("IEC source loaded successfully.")
print("Session established successfully.")

HTTP status: 200
Content type: text/html; charset=utf-8
Response size: 131362 bytes
IEC source loaded successfully.
Session established successfully.
